# Conformal Prediction on Image2Graph Color Dataset

Load the  model

In [ ]:
import torch
from Any2Graph.model import Any2Graph_Model
from ruamel.yaml import YAML

from Models.Any2Graph.Any2Graph.Img2Graph import Img2Graph

yaml = YAML()
config = yaml.load(open("ColoringModel/args.yaml", "r"))
device = "cuda" if torch.cuda.is_available() else "cpu"

weights = torch.load("ColoringModel/best_model")
task = Img2Graph(config)
model = Any2Graph_Model(task, config)

model.load_state_dict(weights)
model.to(device)
model.eval()

print()

Load the dataset

In [ ]:
from Any2Graph.Img2Graph.Coloring.Coloring_Dataset import ColoringDataset

dataset_test = ColoringDataset(
    root_path="Models/Any2Graph/Any2Graph/Img2Graph/Coloring/data/",
    subset="small",
    split="test",
)
dataset_valid = ColoringDataset(
    root_path="Models/Any2Graph/Any2Graph/Img2Graph/Coloring/data/",
    subset="small",
    split="valid",
)
image, graph, idx = dataset_test[0]
print(image.shape)

Precompute sparse (truth, pred) pairs on the test and validation dataset

In [ ]:
from torch.utils.data import DataLoader
from torch import Tensor
from conformal.any2graph import A2GGraph, sparse_from_batch, graph_to_sparse
from torch_geometric.data import Data
from Any2Graph.graphs.custom_graphs_classes import BatchedContinuousGraphs
from tqdm import tqdm


def collate_fn(samples: list[tuple[Tensor, A2GGraph, int]]):
    images = torch.stack([sample[0] for sample in samples])
    graphs = [sample[1] for sample in samples]

    return images, graphs


def precompute_pairs(dataset: ColoringDataset, out_path: str):
    loader = DataLoader(dataset, batch_size=16, collate_fn=collate_fn)

    # Pairs of (truth, preds) sparse graphs for storing
    pairs: list[tuple[Data, Data]] = []

    for images, graphs in tqdm(loader):
        images = images.to(device)
        with torch.no_grad():
            out: BatchedContinuousGraphs = model(images, logits=False).to("cpu")

        preds = sparse_from_batch(out)

        for truth, pred in zip(graphs, preds):
            pairs.append((graph_to_sparse(truth), pred))

    torch.save(pairs, out_path)


precompute_pairs(dataset_test, "color-preds-test.pkl")
precompute_pairs(dataset_valid, "color-preds-valid.pkl")

### Standard Conformal Prediction

The candidate classes for a given ground truth graph
are all the graphs in the dataset which have the same amount of nodes,
and the same amount of each color

In [ ]:
%matplotlib inline
import torch
from torch_geometric.data import Data
from conformal.regression import FixedRegressor
from conformal.fgw import FGW
from conformal.any2graph import equivalence_classes

pairs: list[tuple[Data, Data]] = torch.load("color-preds-test.pkl", weights_only=False)

fgw = FGW(cost="laplacian", prior="uniform")
regressor = FixedRegressor(target=0.9)

Fit the quantile regressor

In [ ]:
from tqdm import tqdm
from conformal.graph import Graph
from conformal.any2graph import sparse_to_graph

distances: list[float] = []

for truth, pred in tqdm(pairs, desc="Computing graph distances"):
    g_truth = Graph.from_a2g(sparse_to_graph(truth))
    g_pred = Graph.from_a2g(sparse_to_graph(pred))
    distances.append(fgw(g_truth, g_pred))


regressor.fit(distances)
print(f"Quantile regressor threshold: {regressor.threshold():.4f}")


Evaluate the model on the validation split

In [ ]:
valid_pairs: list[tuple[Data, Data]] = torch.load(
    "color-preds-valid.pkl", weights_only=False
)
valid_classes = equivalence_classes(dataset_valid)

In [ ]:
from conformal.metrics import Metrics
from conformal.any2graph import graph_class

# Accumulate metrics
correct_coverage: list[bool] = []
candidate_sizes: list[int] = []
conformal_sizes: list[int] = []

for truth, pred in tqdm(valid_pairs):
    a2g_truth = sparse_to_graph(truth)
    a2g_pred = sparse_to_graph(pred)
    g_truth = Graph.from_a2g(a2g_truth)
    g_pred = Graph.from_a2g(a2g_pred)
    equiv_class = graph_class(a2g_truth)
    candidates = valid_classes[equiv_class]

    threshold = regressor.threshold()
    candidate_size = 0
    conformal_size = 0

    # See if the ground truth is in the conformal set
    correct_coverage.append(fgw(g_pred, g_truth) <= threshold)

    for idx in candidates:
        _, a2g_cand, _ = dataset_valid[idx]
        g_cand = Graph.from_a2g(a2g_cand)  # type: ignore

        candidate_size += 1
        if fgw(g_pred, g_cand) <= threshold:
            conformal_size += 1

    candidate_sizes.append(candidate_size)
    conformal_sizes.append(conformal_size)

metrics = Metrics(correct_coverage, candidate_sizes, conformal_sizes)


# save to disk for later analysis
metrics.save("colors-default-metrics.pkl")

print(f"Coverage: {metrics.coverage:.3f}")
print(f"Mean set size: {metrics.mean_set_size:.3f}")
print(f"Median set size: {metrics.median_set_size:.3f}")
print(f"Mean reduction: {metrics.mean_reduction:.3f}")
print(f"Median reduction: {metrics.median_reduction:.3f}")
print(f"Empty set rate: {metrics.empty_rate:.3f}")


### Score Conformal Quantile Regression

Based on the amount of candidates.

Fit the regressor:

In [ ]:
from conformal.regression import CandidateSizeRegressor

regressor = CandidateSizeRegressor(target=0.9)

distances: list[float] = []
candidate_sizes: list[int] = []

test_classes = equivalence_classes(dataset_test)

for truth, pred in tqdm(pairs, desc="Computing graph distances"):
    a2g_truth = sparse_to_graph(truth)
    g_truth = Graph.from_a2g(a2g_truth)
    g_pred = Graph.from_a2g(sparse_to_graph(pred))
    distances.append(fgw(g_truth, g_pred))
    candidate_sizes.append(len(test_classes[graph_class(a2g_truth)]))

regressor.fit(distances, candidate_sizes)
print(f"Quantile regressor threshold: {regressor._threshold:.4f}")

Evaluate the model

In [ ]:
from conformal.metrics import Metrics
from conformal.any2graph import graph_class

# Accumulate metrics
correct_coverage: list[bool] = []
candidate_sizes: list[int] = []
conformal_sizes: list[int] = []

for truth, pred in tqdm(valid_pairs):
    a2g_truth = sparse_to_graph(truth)
    a2g_pred = sparse_to_graph(pred)
    g_truth = Graph.from_a2g(a2g_truth)
    g_pred = Graph.from_a2g(a2g_pred)
    equiv_class = graph_class(a2g_truth)
    candidates = valid_classes[equiv_class]

    threshold = regressor.threshold(len(candidates))
    candidate_size = 0
    conformal_size = 0

    # See if the ground truth is in the conformal set
    correct_coverage.append(fgw(g_pred, g_truth) <= threshold)

    for idx in candidates:
        _, a2g_cand, _ = dataset_valid[idx]
        g_cand = Graph.from_a2g(a2g_cand)  # type: ignore

        candidate_size += 1
        if fgw(g_pred, g_cand) <= threshold:
            conformal_size += 1

    candidate_sizes.append(candidate_size)
    conformal_sizes.append(conformal_size)

metrics = Metrics(correct_coverage, candidate_sizes, conformal_sizes)


# save to disk for later analysis
metrics.save("colors-candidate-metrics.pkl")

print(f"Coverage: {metrics.coverage:.3f}")
print(f"Mean set size: {metrics.mean_set_size:.3f}")
print(f"Median set size: {metrics.median_set_size:.3f}")
print(f"Mean reduction: {metrics.mean_reduction:.3f}")
print(f"Median reduction: {metrics.median_reduction:.3f}")
print(f"Empty set rate: {metrics.empty_rate:.3f}")
